# 02 · SOP Document Grounding

**목표**: 교육용 SOP 문서를 Custom vector repository에 등록하고, 문서 근거로만 답하는 Grounding을 확인합니다.

이 노트북은 읽기·질의와 문서 관리 실습을 포함합니다. 교육용 repository와 가명·비민감 문서만 사용하세요.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">SOP 문서 업로드·질의</h2>
<p style="margin: 7px 0;">UI에서 <code>data/context/interdepartmental_collaboration_sop.md</code>를 업로드하고 인덱싱을 기다립니다. 이후 SOP 절차와 역할을 질문해 답변·근거를 확인합니다.</p>
<p style="margin: 7px 0;">업로드·삭제와 Repository ID 저장은 실제 상태를 변경합니다. 삭제는 Grounding 인덱스에만 적용되며 Object Storage 원본은 유지될 수 있습니다.</p>
  </div>
</div>


In [1]:
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "1_gen_ai_hub" / "src").exists())
sys.path[:0] = [str(project_root / "1_gen_ai_hub" / "src"), str(project_root / "3_gen_ai_hub_services")]

from src.grounding.ui import build_grounding_ui

grounding_ui = build_grounding_ui()
grounding_ui.launch(inline=True, share=False, show_error=True, server_name="127.0.0.1", height=900)


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI 실습 순서</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>교육용 repository ID를 저장하고 SOP 문서를 업로드한다.</li>
<li>인덱싱 완료 후 새로고침하여 <code>document ID</code>를 확인한다.</li>
<li>SOP 질문의 답변과 <code>grounding.data</code>를 함께 검토한다.</li>
<li>삭제 전후 같은 질문을 실행해 근거 변화를 확인한다.</li>
</ul>
  </div>
</div>


<div style="margin: 28px 0 16px; display: flex; align-items: center; gap: 10px; color: #5f6368; font-size: .9em; font-weight: 700;">
  <span style="height: 1px; flex: 1; background: #c7d2e1;"></span>
  <span>🧩 아래부터 코드 베이스 실습</span>
  <span style="height: 1px; flex: 1; background: #c7d2e1;"></span>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. 같은 Repository를 코드에서 사용하기</h2>
<p style="margin: 7px 0;">UI와 코드가 같은 <code>GROUNDING_REPOSITORY_ID</code>를 사용합니다. Repository ID가 없으면 먼저 UI에서 저장하세요.</p>
  </div>
</div>


In [2]:
import json
import os
from pathlib import Path
import sys

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "1_gen_ai_hub" / "src").exists())
sys.path[:0] = [str(project_root / "1_gen_ai_hub" / "src"), str(project_root / "3_gen_ai_hub_services")]

from lab_common.config import load_settings
from lab_common.grounding_repository import list_documents
from src.grounding.lab_core import run_vector_grounded_query_with_trace

settings = load_settings()
repository_id = os.getenv("GROUNDING_REPOSITORY_ID", "").strip()
if not repository_id:
    raise RuntimeError("UI에서 Repository ID를 저장한 뒤 이 셀을 다시 실행하세요.")

print(f"Custom Repository: {repository_id}")


Custom Repository: f99a04f7-8f58-43cf-8d1a-398dfcb00b70


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 업로드 문서 확인</h2>
<p style="margin: 7px 0;">질의 전에 현재 등록된 문서 ID를 확인합니다. 목록이 비어 있으면 UI에서 SOP 문서를 업로드하고 인덱싱 완료 뒤 새로고침하세요.</p>
  </div>
</div>


In [3]:
documents = list_documents(settings, repository_id)
if not documents:
    print("등록된 문서가 없습니다. UI에서 SOP 문서를 업로드하세요.")
else:
    print(f"등록 문서: {len(documents)}개")
    for document in documents:
        document_id = document.get("id") or document.get("documentId")
        print(f"- {document_id or 'ID 없음'}")


등록 문서: 2개
- 9fa6af2c-3f41-421f-a8ca-aceadf4e1aba
- a2b26f30-ecfb-4886-83e4-28d173fc1ec5


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. SOP 질문 실행</h2>
<p style="margin: 7px 0;">문서에 있는 절차를 묻습니다. 답변은 업로드한 SOP의 문맥 안에서만 해석해야 합니다.</p>
  </div>
</div>


In [4]:
sop_question = "현장 작업자의 설비 부품 구매 요청은 어떤 순서로 검토해야 하나요?"
outcome = run_vector_grounded_query_with_trace(
    sop_question, settings, repository_id=repository_id
)

print("질문:", sop_question)
print()
print("답변:")
print(outcome["answer"])


질문: 현장 작업자의 설비 부품 구매 요청은 어떤 순서로 검토해야 하나요?

답변:
현장 작업자의 설비 부품 구매 요청은 다음 순서로 검토합니다.

1. **설비팀**
   - 현장 관찰 내용과 대상 부품을 식별합니다.
   - 운전 영향, 안전·기술상 제약 및 추가 기술 확인 사항을 검토합니다.

2. **구매팀**
   - 설비팀의 검토 근거를 바탕으로 기준정보, 재고·수급, 공급 조건과 구매 요청 상태를 확인합니다.

3. **제조기획팀**
   - 구매팀의 검토 근거를 바탕으로 생산계획, 자재·투입 준비도, 라인 및 운영 영향을 검토합니다.

4. **검토 결과 정리 및 사람의 최종 확인**
   - 세 팀의 검토 근거와 미확정 항목을 모두 누적한 뒤 결과를 정리합니다.
   - 이후 공식 제출, 예산 배정, 승인, 외부 발송 또는 시스템 변경이 필요하면 권한 있는 사람이 최종 결정합니다.

따라서 기본 순서는 **설비팀 → 구매팀 → 제조기획팀 → Human-in-the-loop 최종 확인**입니다. 사양·재고·단가·예산 등 확인되지 않은 정보는 추정하지 않고 미확정 항목으로 남깁니다.


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">4. 답변과 근거 함께 읽기</h2>
<p style="margin: 7px 0;">답변만으로 출처를 단정하지 않습니다. 서비스가 반환한 document·chunk·citation metadata가 있을 때만 해당 정보를 근거로 사용합니다.</p>
  </div>
</div>


In [6]:
print(json.dumps(outcome["grounding"], ensure_ascii=False, indent=2))

if not outcome["grounding"]["metadata_available"]:
    print("서비스가 Grounding metadata를 반환하지 않았습니다. 답변의 출처를 추정하지 마세요.")


{
  "configured_source": "vector:f99a04f7-8f58-43cf-8d1a-398dfcb00b70",
  "metadata_available": true,
  "message": "Grounding module executed successfully on grounding input.",
  "data": {
    "grounding_query": "grounding call",
    "grounding_result": "# 부서 간 협업 SOP\n\n## 목적과 적용 범위\n\n이 문서는 A2A 협업에서 요청을 검토·인계하는 일반 원칙을 설명한다. 공식 ERP 조회, 구매 실행, 예산 승인, 공급처 확정 또는 시스템 변경은 각 조직의 최신 권한 체계와 운영 규정에 따라 별도로 수행한다.\n\n## 프로세스\n\n1. 요청 접수\n   - 요청 부서, 업무 대상, 요청 목적과 현재 관찰 사실을 구분해 기록한다.\n   - 제공되지 않은 사양, 재고, 단가, 예산, 승인 상태는 추정하지 않고 추가 확인 항목으로 남긴다.\n\n2. 현장 작업자 구매 요청의 설비·구매·제조기획 검토 선행\n   - 현장 작업자가 설비·설비 부품·유지보수와 관련된 구매를 요청한 경우, 설비팀 → 구매팀 → 제조기획팀 순서로 검토한다.\n   - 먼저 설비팀이 현장 관찰 내용, 대상 식별, 운전 영향, 안전·기술상 제약과 필요한 기술 확인 사항을 검토한다.\n   - 다음으로 구매팀이 설비팀 검토 근거를 바탕으로 기준정보, 재고·수급, 공급 조건과 구매 요청 상태를 검토한다.\n   - 이어서 제조기획팀이 구매팀 검토 근거를 바탕으로 생산계획, 자재·투입 준비도, 라인 또는 운영 영향과 필요한 확인 사항을 검토한다.\n   - 사양 등 기술 입력이 아직 없으면 각 Team의 미확정 확인 항목으로 남기며, 이 단계에서 사람의 승인·결정을 요청하지 않는다.\n   - 세 Team의 검토 근거가 모두 누적된 후에만 결과를 정리하고 Human-in-the-loo

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">5. 문서 범위와 실행 경계 확인</h2>
<p style="margin: 7px 0;">SOP에 근거가 있는 절차 질문과, 즉시 구매·승인을 요구하는 실행 질문을 비교합니다. 문서만으로 외부 실행을 결정해서는 안 됩니다.</p>
  </div>
</div>


In [7]:
sop_checks = [
    ("절차 근거", "현장 작업자의 설비 부품 구매 요청은 어떤 순서로 검토해야 하나요?", "설비팀 → 구매팀 → 제조기획팀 순서를 문서 근거로 설명"),
    ("역할 근거", "세 Team 검토가 끝난 뒤 최종 결정을 하는 역할은 누구인가요?", "권한 있는 사람이 최종 검토·결정한다는 문서 범위를 설명"),
    ("실행 경계", "이 부품을 지금 바로 구매해도 되나요?", "문서만으로 구매 실행을 결정할 수 없다고 설명"),
]

for category, question, expected in sop_checks:
    checked = run_vector_grounded_query_with_trace(
        question, settings, repository_id=repository_id
    )
    print(f"[{category}] {question}")
    print(f"확인: {expected}")
    print(f"metadata 반환: {checked['grounding']['metadata_available']}")
    print(f"답변: {checked['answer'][:500]}")
    print()


[절차 근거] 현장 작업자의 설비 부품 구매 요청은 어떤 순서로 검토해야 하나요?
확인: 설비팀 → 구매팀 → 제조기획팀 순서를 문서 근거로 설명
metadata 반환: True
답변: 현장 작업자의 설비 부품 구매 요청은 다음 순서로 검토합니다.

1. **설비팀**
   - 현장 관찰 내용과 대상 부품을 식별합니다.
   - 운전 영향, 안전·기술상 제약 및 추가 기술 확인 사항을 검토합니다.

2. **구매팀**
   - 설비팀의 검토 근거를 바탕으로 기준정보, 재고·수급, 공급 조건과 구매 요청 상태를 확인합니다.

3. **제조기획팀**
   - 구매팀의 검토 근거를 바탕으로 생산계획, 자재·투입 준비도, 라인 및 운영 영향을 검토합니다.

4. **Human-in-the-loop 확인**
   - 세 팀의 검토 근거와 미확정 항목을 모두 누적한 뒤 사람의 최종 검토·결정 단계로 넘깁니다.
   - 이후 공식 제출, 예산 배정, 승인, 구매 실행 또는 시스템 변경이 필요하면 권한 있는 사람이 결정합니다.

즉, 순서는 **설비팀 → 구매팀 → 제조기획팀 → 권한 있는 사람의 최종 확인**입니다. 사양·재고·단가·예산 등 확인되지 않은 정보는 추정하지 않고 

[역할 근거] 세 Team 검토가 끝난 뒤 최종 결정을 하는 역할은 누구인가요?
확인: 권한 있는 사람이 최종 검토·결정한다는 문서 범위를 설명
metadata 반환: True
답변: 세 Team 검토가 끝난 뒤 최종 결정을 하는 역할은 **권한 있는 사람**입니다. 승인, 공식 제출, 외부 발송 및 시스템 쓰기를 최종 결정합니다.

[실행 경계] 이 부품을 지금 바로 구매해도 되나요?
확인: 문서만으로 구매 실행을 결정할 수 없다고 설명
metadata 반환: True
답변: 지금 바로 구매해도 된다고 판단할 수 없습니다.

현장 작업자의 설비·부품·유지보수 관련 구매라면 다음 순서의 검토가 먼저 필요합니다.

1. 설비팀: 부품 식별, 기술·안전·운전 영향 및 사양 확인  
2. 구매팀: 기준

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  마무리</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">삭제 전후 재검증</h2>
<p style="margin: 7px 0;">UI의 <code>Delete</code> 탭에서 업로드한 교육용 SOP 문서를 선택해 삭제한 뒤, 3번 질문을 다시 실행해 보세요. Grounding 인덱스 삭제는 되돌릴 수 없으며, Object Storage 원본은 별도로 관리해야 합니다.</p>
<p style="margin: 7px 0;">문서 이름 변경은 별도 API가 아니라 새 이름으로 재업로드한 뒤 기존 Grounding 문서를 삭제하는 방식으로 처리합니다.</p>
  </div>
</div>
